# FPA Archive — JSON Content Tests

This notebook checks the JSON files produced for FPA Archive are correct. It reads from `gold_appeals_with_json.JSON_content` and verifies the values against silver and bronze sources independently — so it catches bugs the pipeline itself wouldn't notice.

Each section below runs one test. Results land in a CSV plus `test_automation_runs2` / `test_automation_results2` / `test_automation_perfresults2` so they show up in the dashboards. Every result row carries a plain-English `description` explaining what was checked.

In [0]:
########################
# test Setup
########################

state_under_test = "archive_fpa_json"
run_tag_default = "fpa_json_content"

dbutils.widgets.text("run_tag", run_tag_default)
dbutils.widgets.text("hive_schema", "ariadm_arm_fpa")
dbutils.widgets.text("sample_size", "20")

run_tag     = dbutils.widgets.get("run_tag") or run_tag_default
hive_schema = dbutils.widgets.get("hive_schema") or "ariadm_arm_fpa"
sample_size = int(dbutils.widgets.get("sample_size") or 20)

In [0]:
import os
import sys
import json as _json
import random
import time as perf_time
from datetime import datetime

from pyspark.sql import functions as F
from pyspark.sql.functions import col

run_user = dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get()
test_results_path = "/Workspace/Users/" + run_user + "/Results/Archive_FPA_Tests"
os.makedirs(test_results_path, exist_ok=True)

for _mod in list(sys.modules.keys()):
    if _mod.startswith("Test_Functions") or _mod == "models.test_result":
        del sys.modules[_mod]

from models.test_result import TestResult
from Test_Functions.report_helpers import (
    build_report_folder,
    write_csv, write_xlsx, write_html,
    count_by_status,
    create_run, create_results, create_perf_results,
)

print(f"User: {run_user}")
print(f"state_under_test: {state_under_test}")
print(f"run_tag: {run_tag}")
print(f"hive_schema: {hive_schema}")
print(f"sample_size: {sample_size}")
print(f"Results folder root: {test_results_path}")

In [0]:
#######################
# Spark Config
#######################
config_path = "dbfs:/configs/config.json"
config = spark.read.option("multiline", "true").json(config_path)
first_row = config.first()
env = first_row["env"].strip().lower()
lz_key = first_row["lz_key"].strip().lower()
keyvault_name = f"ingest{lz_key}-meta002-{env}"

client_secret = dbutils.secrets.get(scope=keyvault_name, key="SERVICE-PRINCIPLE-CLIENT-SECRET")
tenant_id     = dbutils.secrets.get(scope=keyvault_name, key="SERVICE-PRINCIPLE-TENANT-ID")
client_id     = dbutils.secrets.get(scope=keyvault_name, key="SERVICE-PRINCIPLE-CLIENT-ID")

curated_storage    = f"ingest{lz_key}curated{env}"
checkpoint_storage = f"ingest{lz_key}xcutting{env}"
raw_storage        = f"ingest{lz_key}raw{env}"
landing_storage    = f"ingest{lz_key}landing{env}"
external_storage   = f"ingest{lz_key}external{env}"

for storage in (curated_storage, checkpoint_storage, raw_storage, landing_storage, external_storage):
    spark.conf.set(f"fs.azure.account.auth.type.{storage}.dfs.core.windows.net", "OAuth")
    spark.conf.set(f"fs.azure.account.oauth.provider.type.{storage}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
    spark.conf.set(f"fs.azure.account.oauth2.client.id.{storage}.dfs.core.windows.net", client_id)
    spark.conf.set(f"fs.azure.account.oauth2.client.secret.{storage}.dfs.core.windows.net", client_secret)
    spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{storage}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")

archive_silver_path = f"abfss://silver@ingest{lz_key}curated{env}.dfs.core.windows.net/ARIADM/ARM/APPEALS/ARIAFPA/"
archive_gold_path   = f"abfss://gold@ingest{lz_key}curated{env}.dfs.core.windows.net/ARIADM/ARM/APPEALS/ARIAFPA/JSON/"

print(f"env={env}  lz_key={lz_key}")
print(f"silver path: {archive_silver_path}")
print(f"gold JSON path: {archive_gold_path}")

## Load the gold and silver tables

The cell below loads:
- **gold_appeals_with_json** — the table that contains the JSON_content column we're checking.
- Each silver source table we'll need (appealcase, applicant, dependent, history, status, list, transaction, documents, appealcategory, link) plus optional silvers for less-used arrays.
- **Bronze tables** if available — used by Tests 7-10 to verify pass-through and round-trip values.

In [0]:
def _try_load(table_name):
    try:
        return spark.table(f"hive_metastore.{hive_schema}.{table_name}")
    except Exception as _e:
        print(f"{table_name} not loadable: {_e}")
        return None

gold_json = spark.table(f"hive_metastore.{hive_schema}.gold_appeals_with_json").alias("g")

silver_appealcase = spark.table(f"hive_metastore.{hive_schema}.silver_appealcase_detail")
silver_applicant  = spark.table(f"hive_metastore.{hive_schema}.silver_applicant_detail")
silver_dependent  = spark.table(f"hive_metastore.{hive_schema}.silver_dependent_detail")
silver_history    = spark.table(f"hive_metastore.{hive_schema}.silver_history_detail")
silver_status     = spark.table(f"hive_metastore.{hive_schema}.silver_status_detail")
silver_list       = spark.table(f"hive_metastore.{hive_schema}.silver_list_detail")
silver_transaction= spark.table(f"hive_metastore.{hive_schema}.silver_transaction_detail")
silver_documents  = spark.table(f"hive_metastore.{hive_schema}.silver_documents_detail")
silver_apl_cat    = spark.table(f"hive_metastore.{hive_schema}.silver_appealcategory_detail")
silver_link       = spark.table(f"hive_metastore.{hive_schema}.silver_link_detail")

# Optional silvers — used by the array-absence verifier. Best-effort.
silver_appealgrounds      = _try_load("silver_appealgrounds_detail")
silver_appealtypecategory = _try_load("silver_appealtypecategory_detail")
silver_bfdairy            = _try_load("silver_dfdairy_detail")
silver_reviewspecdir      = _try_load("silver_reviewspecificdirection_detail")
silver_statusdecisiontype = _try_load("silver_statusdecisiontype_detail")

# Bronze (optional — used by the bronze-source tests). Best-effort load each.
bronze_appealcase  = _try_load("bronze_appealcase_cr_cs_ca_fl_cres_mr_res_lang")
bronze_applicant   = _try_load("bronze_appealcase_ca_apt_country_detc")
bronze_history     = _try_load("bronze_appealcase_history_users")
bronze_status      = _try_load("bronze_appealcase_status_sc_ra_cs")
bronze_list        = _try_load("bronze_appealcase_cl_ht_list_lt_hc_c_ls_adj")
bronze_transaction = _try_load("bronze_appealcase_t_tt_ts_tm")
bronze_documents   = _try_load("bronze_appealcase_dr_rd")

SILVER_BY_NAME = {
    "silver_history":     silver_history,
    "silver_status":      silver_status,
    "silver_list":        silver_list,
    "silver_transaction": silver_transaction,
    "silver_documents":   silver_documents,
    "silver_apl_cat":     silver_apl_cat,
    "silver_link":        silver_link,
}

BRONZE_BY_NAME = {
    "bronze_history":     bronze_history,
    "bronze_status":      bronze_status,
    "bronze_list":        bronze_list,
    "bronze_transaction": bronze_transaction,
    "bronze_documents":   bronze_documents,
}

# Cache gold table — every test reads it. Without this the same table is re-scanned ~20 times per run.
gold_json = gold_json.cache()
gold_count = gold_json.count()    # forces materialization
print(f"gold_appeals_with_json rows: {gold_count} (cached)")

## Pick a representative sample of cases

Tests 4, 5, 7-11 only check a sample of cases (cheaper than running over the whole table). The cell below deliberately picks cases that exercise interesting edge cases — the one with the most history rows, one with dependents, one with a NULL AppellantTitle, one with linked cases — then fills the rest with random cases. Seeded so reruns are reproducible.

In [0]:
run_start_datetime = datetime.utcnow()
all_test_results = []
perf_timings = []
t0 = perf_time.time()

def _run_timed(label, fn, *args, **kwargs):
    """Run a test function and capture wall-clock + how many TestResult rows it added.

    Inline-prints a one-line progress marker so you can see slow tests as they run.
    Captures the elapsed time and result_count into perf_timings, which gets
    persisted to test_automation_perfresults2 at the end of the notebook."""
    before = len(all_test_results)
    _t0 = perf_time.time()
    try:
        return fn(*args, **kwargs)
    finally:
        elapsed = perf_time.time() - _t0
        added = len(all_test_results) - before
        perf_timings.append({"test_name": label, "elapsed_seconds": elapsed, "result_count": added})
        print(f"  [{elapsed:6.2f}s, +{added:4d} results] {label}")

def _safe_max_by_count(df, label):
    """Return the CaseNo with the highest per-CaseNo row count, or None."""
    try:
        top = df.groupBy("CaseNo").count().orderBy(col("count").desc()).limit(1).collect()
        return top[0].CaseNo if top else None
    except Exception as e:
        print(f"stratify[{label}] failed: {e}")
        return None

def _safe_first_where(df, condition, label):
    try:
        rows = df.filter(condition).select("CaseNo").limit(1).collect()
        return rows[0].CaseNo if rows else None
    except Exception as e:
        print(f"stratify[{label}] failed: {e}")
        return None

def pick_stratified_sample(gold_json_df, n):
    seeds = []
    seeds.append(_safe_max_by_count(silver_history,     "max_history"))
    seeds.append(_safe_max_by_count(silver_status,      "max_status"))
    seeds.append(_safe_max_by_count(silver_list,        "max_list"))
    seeds.append(_safe_max_by_count(silver_transaction, "max_transaction"))
    seeds.append(_safe_first_where(silver_dependent, F.lit(True), "has_dependents"))
    seeds.append(_safe_first_where(silver_link,      F.lit(True), "has_linked_cases"))
    seeds.append(_safe_first_where(silver_applicant, col("AppellantTitle").isNull(), "null_title"))
    seeds.append(_safe_first_where(silver_appealcase, col("FileLocationNote").isNull(), "null_filelocnote"))

    seeds = list(dict.fromkeys([s for s in seeds if s]))

    if seeds:
        in_gold = [r.CaseNo for r in gold_json_df.filter(col("CaseNo").isin(*seeds)).select("CaseNo").collect()]
    else:
        in_gold = []

    needed = max(0, n - len(in_gold))
    if needed > 0:
        if in_gold:
            fill_df = gold_json_df.filter(~col("CaseNo").isin(*in_gold))
        else:
            fill_df = gold_json_df
        fill = [r.CaseNo for r in fill_df.orderBy(F.rand(seed=42)).select("CaseNo").limit(needed).collect()]
        in_gold.extend(fill)
    return in_gold[:n]

SAMPLE_CASES = pick_stratified_sample(gold_json, sample_size)
print(f"Sample ({len(SAMPLE_CASES)} cases): {SAMPLE_CASES[:5]}{'...' if len(SAMPLE_CASES) > 5 else ''}")

# Pre-load JSON content for the sample — used by several driver-side tests
SAMPLE_JSON = {}
for r in gold_json.filter(col("CaseNo").isin(*SAMPLE_CASES)).select("CaseNo", "JSON_content").collect():
    try:
        SAMPLE_JSON[r.CaseNo] = _json.loads(r.JSON_content) if r.JSON_content else None
    except Exception as _e:
        SAMPLE_JSON[r.CaseNo] = None
        print(f"sample JSON parse failed for {r.CaseNo}: {_e}")
print(f"Pre-loaded {sum(1 for v in SAMPLE_JSON.values() if v is not None)} parsed sample JSONs.")

## Test 1 — Are the JSON files well-formed?

The cell below loops over every row in the gold table and checks three things:
- the `JSON_content` string actually parses as JSON;
- the top-level `CaseNo` inside the JSON matches the row's `CaseNo` column;
- no row's `JSON_content` starts with `'failure'` (pipeline error markers that should have been dropped before gold).

In [0]:
def test_json_parseable_and_caseno_matches(gold_json_df):
    try:
        parsed = gold_json_df.select(
            col("CaseNo").alias("row_case"),
            F.get_json_object(col("JSON_content"), "$.CaseNo").alias("json_case"),
            col("JSON_content").isNull().alias("is_null_content"),
        )
        null_content = parsed.filter(col("is_null_content")).count()
        # If JSON is malformed, get_json_object returns null; CaseNo is required, so null here = parse failure
        unparseable = parsed.filter(~col("is_null_content") & col("json_case").isNull()).count()
        mismatched  = parsed.filter(col("json_case").isNotNull() & (col("json_case") != col("row_case"))).count()

        for field, n in [("null_JSON_content", null_content),
                         ("unparseable_or_missing_CaseNo", unparseable),
                         ("CaseNo_mismatch_row_vs_json", mismatched)]:
            status = "PASS" if n == 0 else "FAIL"
            msg = "" if n == 0 else f"{n} rows"
            all_test_results.append(TestResult(test_name="test_json_parseable_and_caseno_matches",
                                               test_field=field, test_from_state=state_under_test,
                                               status=status, message=msg))
    except Exception as e:
        all_test_results.append(TestResult(test_name="test_json_parseable_and_caseno_matches",
                                           test_field="ALL", test_from_state=state_under_test,
                                           status="ERROR", message=f"{type(e).__name__}: {e}"))

def test_no_failure_json_content(gold_json_df):
    try:
        n = gold_json_df.filter(F.lower(col("JSON_content")).like("failure%")).count()
        status = "PASS" if n == 0 else "FAIL"
        msg = "" if n == 0 else f"{n} rows have failure% JSON"
        all_test_results.append(TestResult(test_name="test_no_failure_json_content",
                                           test_field="ALL", test_from_state=state_under_test,
                                           status=status, message=msg))
    except Exception as e:
        all_test_results.append(TestResult(test_name="test_no_failure_json_content",
                                           test_field="ALL", test_from_state=state_under_test,
                                           status="ERROR", message=f"{type(e).__name__}: {e}"))

_run_timed("test_json_parseable_and_caseno_matches", test_json_parseable_and_caseno_matches, gold_json)
_run_timed("test_no_failure_json_content", test_no_failure_json_content, gold_json)

## Test 2 — Are the required top-level keys present?

Splits into three checks:
- **Required scalars** (`CaseNo`, `CasePrefix`, `CaseYear`, `AppealTypeId`, `currentstatus`, `AppellantId`, `AppellantFullName`) — these MUST be on every row. A missing one is a FAIL with the offending CaseNo.
- **Informational scalars** (`DateLodged`) — usually populated but legitimately NULL on some legacy/pre-2000 cases. Reported with a count, never FAIL.
- **Expected array keys** (`HistoryDetails`, `ListDetails`, `TransactionDetails`, etc.) — these MAY be absent on rows where the case genuinely has no source data; reported with a count, not a FAIL. Misspellings (`BFDairyDetails`, `AppealTypeCategorieDetails`, `ReviewSpecficDirectionDetails`) are part of the contract — kept as-is.

In [0]:
# Scalars: must be non-null on every row (per LLD's mandatory metadata + minimum case-header contract).
REQUIRED_SCALAR_KEYS = [
    "CaseNo", "CasePrefix", "CaseYear", "AppealTypeId",
    "currentstatus", "AppellantId", "AppellantFullName",
]

# Informational scalars: usually populated but legitimately NULL on some legacy data
# (e.g. DateLodged is missing on pre-2000 cases that pre-date that field). Reported with
# a count, never FAIL. The LLD doesn't include these on the mandatory list.
INFORMATIONAL_SCALAR_KEYS = [
    "DateLodged",
]

# Arrays: Spark's to_json(struct(*), ignoreNullFields=true) OMITS the key entirely
# when the case has zero source rows for that array. Informational only — reported
# with a "present on N/M rows" count rather than FAILed.
EXPECTED_ARRAY_KEYS = [
    "ListDetails", "HistoryDetails", "StatusDetails",
    "AppealCategoryDetails", "TransactionDetails", "DocumentDetails",
    "AppealTypeCategorieDetails", "AppealGroundsDetails",
    "StatusDecisionTypeDetails", "BFDairyDetails", "ReviewSpecficDirectionDetails",
]

def test_required_top_level_keys(gold_json_df):
    total = gold_json_df.count()
    for k in REQUIRED_SCALAR_KEYS:
        try:
            missing_df = gold_json_df.filter(
                F.get_json_object(col("JSON_content"), f"$.{k}").isNull()
            )
            missing = missing_df.count()
            if missing == 0:
                all_test_results.append(TestResult(test_name="test_required_scalar_keys",
                                                   test_field=k, test_from_state=state_under_test,
                                                   status="PASS", message=""))
            else:
                examples = [r.CaseNo for r in missing_df.select("CaseNo").limit(5).collect()]
                all_test_results.append(TestResult(test_name="test_required_scalar_keys",
                                                   test_field=k, test_from_state=state_under_test,
                                                   status="FAIL",
                                                   message=f"{missing} rows missing key; examples: {examples}"))
        except Exception as e:
            all_test_results.append(TestResult(test_name="test_required_scalar_keys",
                                               test_field=k, test_from_state=state_under_test,
                                               status="ERROR", message=f"{type(e).__name__}: {e}"))

    for k in INFORMATIONAL_SCALAR_KEYS:
        try:
            missing_df = gold_json_df.filter(
                F.get_json_object(col("JSON_content"), f"$.{k}").isNull()
            )
            missing = missing_df.count()
            present = total - missing
            examples = [r.CaseNo for r in missing_df.select("CaseNo").limit(3).collect()] if missing else []
            msg = f"present on {present}/{total} rows; absent={missing}"
            if examples:
                msg += f"; examples of absent: {examples}"
            all_test_results.append(TestResult(test_name="test_informational_scalar_keys",
                                               test_field=k, test_from_state=state_under_test,
                                               status="PASS", message=msg))
        except Exception as e:
            all_test_results.append(TestResult(test_name="test_informational_scalar_keys",
                                               test_field=k, test_from_state=state_under_test,
                                               status="ERROR", message=f"{type(e).__name__}: {e}"))

    for k in EXPECTED_ARRAY_KEYS:
        try:
            missing = gold_json_df.filter(
                F.get_json_object(col("JSON_content"), f"$.{k}").isNull()
            ).count()
            present = total - missing
            all_test_results.append(TestResult(test_name="test_expected_array_keys",
                                               test_field=k, test_from_state=state_under_test,
                                               status="PASS",
                                               message=f"present on {present}/{total} rows; absent={missing}"))
        except Exception as e:
            all_test_results.append(TestResult(test_name="test_expected_array_keys",
                                               test_field=k, test_from_state=state_under_test,
                                               status="ERROR", message=f"{type(e).__name__}: {e}"))

_run_timed("test_required_top_level_keys", test_required_top_level_keys, gold_json)

## Test 3 — Are missing array keys actually empty in silver?

Test 2 reports counts of rows where an array key is missing — but that doesn't tell us whether those absences are correct. The cell below settles it: for each absent array key, we look up the case in the silver source table.
- If silver also has zero rows for that case → expected absent (PASS).
- If silver has rows that the JSON dropped → real data loss (FAIL, with offending CaseNo examples).

In [0]:
# Mapping: JSON array key -> silver source DataFrame (None if not loadable in this env)
ABSENCE_SILVER_SOURCE = {
    "ListDetails":                    silver_list,
    "HistoryDetails":                 silver_history,
    "StatusDetails":                  silver_status,
    "TransactionDetails":             silver_transaction,
    "DocumentDetails":                silver_documents,            # pre-aggregated; null-array == no rows
    "AppealCategoryDetails":          silver_apl_cat,
    "AppealGroundsDetails":           silver_appealgrounds,
    "AppealTypeCategorieDetails":     silver_appealtypecategory,
    "BFDairyDetails":                 silver_bfdairy,
    "ReviewSpecficDirectionDetails":  silver_reviewspecdir,
    "StatusDecisionTypeDetails":      silver_statusdecisiontype,
}

# Silver tables that are pre-aggregated (single row per case with a nested array column).
# Same shape we already handled in T4 / T12. "rows" here means "non-null array".
ABSENCE_SILVER_AGG_COL = {
    "DocumentDetails": "DocumentDetails",
}

def test_array_absences_match_silver(gold_json_df):
    # Single pass over gold: compute absence flag per array key in one select, then cache.
    # Without this each key would re-scan gold (11 full scans dominated FTA's 60-min runtime).
    keys_to_check = [k for k, v in ABSENCE_SILVER_SOURCE.items() if v is not None]
    flag_cols = [F.get_json_object(col("JSON_content"), f"$.{k}").isNull().alias(f"absent__{k}") for k in keys_to_check]
    absent_flags = gold_json_df.select(col("CaseNo"), *flag_cols).cache()
    absent_flags.count()  # materialise

    # One distinct-CaseNo scan per UNIQUE silver_df, cached by id so shared silvers scan once.
    # Avoids the giant filter(col.isin(*N_thousand_cases)) OR-expression that dominated runtime.
    silver_cases_by_id = {}

    for json_key, silver_df in ABSENCE_SILVER_SOURCE.items():
        if silver_df is None:
            all_test_results.append(TestResult(test_name="test_array_absences_match_silver",
                                               test_field=json_key, test_from_state=state_under_test,
                                               status="NO_DATA", message="silver source not loadable"))
            continue
        try:
            absent_df = absent_flags.filter(col(f"absent__{json_key}")).select(col("CaseNo").alias("absent_case"))
            absent_cases = [r.absent_case for r in absent_df.collect()]
            if not absent_cases:
                all_test_results.append(TestResult(test_name="test_array_absences_match_silver",
                                                   test_field=json_key, test_from_state=state_under_test,
                                                   status="PASS", message="key present on every row"))
                continue

            # Find which absent cases ACTUALLY have silver rows (= real data loss)
            sid = id(silver_df)
            if sid not in silver_cases_by_id:
                agg_col = ABSENCE_SILVER_AGG_COL.get(json_key)
                if agg_col and agg_col in silver_df.columns:
                    # pre-aggregated silver: "has data" means non-null array of length > 0
                    rows = silver_df.filter(F.coalesce(F.size(col(agg_col)), F.lit(0)) > 0)                                     .select("CaseNo").distinct().collect()
                else:
                    rows = silver_df.select("CaseNo").distinct().collect()
                silver_cases_by_id[sid] = {r.CaseNo for r in rows}
            silver_cases = silver_cases_by_id[sid]
            # Intersect in Python — no giant isin OR-expression.
            real_loss_cases = [c for c in absent_cases if c in silver_cases]

            n_absent = len(absent_cases)
            n_loss = len(real_loss_cases)
            n_expected = n_absent - n_loss

            if n_loss == 0:
                status = "PASS"
                msg = f"all {n_absent} absences confirmed empty in silver"
            else:
                status = "FAIL"
                msg = f"{n_loss}/{n_absent} absences have silver rows (real data loss); examples: {real_loss_cases[:5]}"
            all_test_results.append(TestResult(test_name="test_array_absences_match_silver",
                                               test_field=json_key, test_from_state=state_under_test,
                                               status=status, message=msg))
        except Exception as e:
            all_test_results.append(TestResult(test_name="test_array_absences_match_silver",
                                               test_field=json_key, test_from_state=state_under_test,
                                               status="ERROR", message=f"{type(e).__name__}: {e}"))

_run_timed("test_array_absences_match_silver", test_array_absences_match_silver, gold_json)

## Test 4 — Do JSON arrays match silver counts?

For each sampled case, the cell below counts silver rows independently (per CaseNo, our own filter — NOT reusing the dev's 24-way LEFT JOIN) and compares that count to the length of the matching JSON array. Catches the pipeline silently dropping or duplicating rows on the way to gold.

Checks 6 arrays: HistoryDetails, StatusDetails, ListDetails, TransactionDetails, DocumentDetails, AppealCategoryDetails.

In [0]:
ARRAY_VS_SILVER = [
    ("HistoryDetails",        "silver_history"),
    ("StatusDetails",         "silver_status"),
    ("ListDetails",           "silver_list"),
    ("TransactionDetails",    "silver_transaction"),
    ("DocumentDetails",       "silver_documents"),
    ("AppealCategoryDetails", "silver_apl_cat"),
]

# silver_documents_detail is one row per case with a nested DocumentDetails array,
# unlike the other silver_*_detail tables which are one row per array entry.
# For documents we count via size(DocumentDetails) instead of row count.
SILVER_AGGREGATED_ARRAY_COL = {
    "silver_documents": "DocumentDetails",
}

def test_array_cardinality_vs_silver():
    if not SAMPLE_CASES:
        all_test_results.append(TestResult(test_name="test_array_cardinality_vs_silver",
                                           test_field="ALL", test_from_state=state_under_test,
                                           status="PASS", message="not applicable: no sample available (gold may be empty for this segment)"))
        return
    silver_counts = {}
    for json_key, silver_var in ARRAY_VS_SILVER:
        try:
            df = SILVER_BY_NAME[silver_var]
            agg_col = SILVER_AGGREGATED_ARRAY_COL.get(silver_var)
            if agg_col and agg_col in df.columns:
                rows = df.filter(col("CaseNo").isin(*SAMPLE_CASES))                          .select("CaseNo", F.coalesce(F.size(agg_col), F.lit(0)).alias("count")).collect()
            else:
                rows = df.filter(col("CaseNo").isin(*SAMPLE_CASES)).groupBy("CaseNo").count().collect()
            silver_counts[silver_var] = {r.CaseNo: r["count"] for r in rows}
        except Exception as e:
            silver_counts[silver_var] = None
            print(f"silver count for {silver_var} failed: {e}")

    for case_no in SAMPLE_CASES:
        obj = SAMPLE_JSON.get(case_no)
        if obj is None:
            for json_key, _ in ARRAY_VS_SILVER:
                all_test_results.append(TestResult(test_name="test_array_cardinality_vs_silver",
                                                   test_field=f"{case_no}:{json_key}",
                                                   test_from_state=state_under_test,
                                                   status="ERROR", message="sample JSON unparseable"))
            continue
        for json_key, silver_var in ARRAY_VS_SILVER:
            try:
                json_arr = obj.get(json_key) or []
                json_n = len(json_arr) if isinstance(json_arr, list) else -1
                sc = silver_counts.get(silver_var)
                if sc is None:
                    all_test_results.append(TestResult(test_name="test_array_cardinality_vs_silver",
                                                       test_field=f"{case_no}:{json_key}",
                                                       test_from_state=state_under_test,
                                                       status="ERROR", message="silver count unavailable"))
                    continue
                silver_n = sc.get(case_no, 0)
                ok = json_n == silver_n
                all_test_results.append(TestResult(test_name="test_array_cardinality_vs_silver",
                                                   test_field=f"{case_no}:{json_key}",
                                                   test_from_state=state_under_test,
                                                   status=("PASS" if ok else "FAIL"),
                                                   message=("" if ok else f"json={json_n} silver={silver_n}")))
            except Exception as e:
                all_test_results.append(TestResult(test_name="test_array_cardinality_vs_silver",
                                                   test_field=f"{case_no}:{json_key}",
                                                   test_from_state=state_under_test,
                                                   status="ERROR", message=f"{type(e).__name__}: {e}"))

_run_timed("test_array_cardinality_vs_silver", test_array_cardinality_vs_silver)

## Test 5 — Are primary appellants and dependents in the right place?

For each sampled case, the cell below checks two things:
- The `AppellantId` shown in the JSON's top-level appellant fields should appear in `silver_applicant_detail` (which filters to primary appellants only) and NOT in `silver_dependent_detail`.
- Every entry in the JSON's `DependentDetails` array should come from `silver_dependent_detail` and NOT `silver_applicant_detail`.

In [0]:
def test_primary_appellant_partition():
    for case_no in SAMPLE_CASES:
        obj = SAMPLE_JSON.get(case_no)
        if obj is None:
            continue
        try:
            json_app_id = obj.get("AppellantId")
            prim_ids = [r.AppellantId for r in silver_applicant.filter(col("CaseNo") == case_no).select("AppellantId").collect()]
            dep_ids  = [r.AppellantId for r in silver_dependent.filter(col("CaseNo") == case_no).select("AppellantId").collect()]

            in_primary = json_app_id in prim_ids
            in_dependent = json_app_id in dep_ids
            ok = in_primary and not in_dependent
            if ok:
                msg = ""
            else:
                msg = f"json_app_id={json_app_id} primary_ids={prim_ids[:5]} dependent_ids={dep_ids[:5]}"
            all_test_results.append(TestResult(test_name="test_primary_appellant_partition",
                                               test_field=case_no, test_from_state=state_under_test,
                                               status=("PASS" if ok else "FAIL"), message=msg))
        except Exception as e:
            all_test_results.append(TestResult(test_name="test_primary_appellant_partition",
                                               test_field=case_no, test_from_state=state_under_test,
                                               status="ERROR", message=f"{type(e).__name__}: {e}"))

def test_dependent_partition():
    for case_no in SAMPLE_CASES:
        obj = SAMPLE_JSON.get(case_no)
        if obj is None:
            continue
        try:
            deps = obj.get("DependentDetails") or []
            if not deps:
                all_test_results.append(TestResult(test_name="test_dependent_partition",
                                                   test_field=case_no, test_from_state=state_under_test,
                                                   status="PASS", message="no dependents"))
                continue
            json_dep_ids = [d.get("AppellantId") for d in deps if isinstance(d, dict)]
            prim_ids = set(r.AppellantId for r in silver_applicant.filter(col("CaseNo") == case_no).select("AppellantId").collect())
            dep_ids  = set(r.AppellantId for r in silver_dependent.filter(col("CaseNo") == case_no).select("AppellantId").collect())

            wrong_partition = [i for i in json_dep_ids if i in prim_ids]
            not_in_silver   = [i for i in json_dep_ids if i not in dep_ids]
            ok = not wrong_partition and not not_in_silver
            if ok:
                msg = f"verified {len(json_dep_ids)} dependents"
            else:
                msg = f"in_primary={wrong_partition[:5]} not_in_silver_dep={not_in_silver[:5]}"
            all_test_results.append(TestResult(test_name="test_dependent_partition",
                                               test_field=case_no, test_from_state=state_under_test,
                                               status=("PASS" if ok else "FAIL"), message=msg))
        except Exception as e:
            all_test_results.append(TestResult(test_name="test_dependent_partition",
                                               test_field=case_no, test_from_state=state_under_test,
                                               status="ERROR", message=f"{type(e).__name__}: {e}"))

_run_timed("test_primary_appellant_partition", test_primary_appellant_partition)
_run_timed("test_dependent_partition", test_dependent_partition)

## Test 6 — Are coded values translated to the right labels?

Some fields in the source data are numeric codes that silver translates to text labels (e.g. `Interpreter` 1→`'YES'`, 2→`'NO'`; `PubliclyFunded` `true`→`'checked'`, `false`→`'disabled'`). The cell below collects every distinct JSON value at each WHEN-mapped path across the whole gold table and checks it's in the documented set of allowed labels. Catches raw codes leaking through unmapped AND drift in the mapping itself.

In [0]:
# (path, allowed-values-set). None means "allow null too".
WHEN_MAP_RANGES = {
    "$.Interpreter":         {"YES", "NO", "", None},
    "$.VisitVisaType":       {"", "On Papers", "Oral Hearing", None},
    "$.AdditionalGrounds":   {"", "YES", "NO", None},
    "$.AppealCategories":    {"YES", "NO", "", None},
    "$.HumanRights":         {"", "YES", "NO", None},
    "$.Detained":            {"HMP", "IRC", "No", "Other", "", None},
    "$.AppealReceivedBy":    {"Fax", "Fax and Post", "Hand", "Post", "On-Line", "", None},
    "$.CRRespondent":        {"respondent", "embassy", "POU", "", None},
    "$.LSCCommission":       {"", "England & Wales", "Northern Ireland", "Scotland", None},
    "$.CourtPreference":     {"", "All-Male", "All-Female", None},
    "$.PubliclyFunded":      {"checked", "disabled", None},
    "$.NonStandardSCPeriod": {"checked", "disabled", None},
    "$.InCamera":            {"checked", "disabled", None},
    "$.SecureCourtRequired": {"checked", "disabled", None},
    "$.OutOfTimeIssue":      {"checked", "disabled", None},
    "$.validityIssues":      {"checked", "disabled", None},
    "$.FileInStatutoryClosure":   {"checked", "disabled", None},
    "$.DateOfNextListedHearing":  {"checked", "disabled", None},
    "$.Authorised":          {"checked", "disabled", None},
}

def test_when_map_value_ranges(gold_json_df):
    for path, allowed in WHEN_MAP_RANGES.items():
        try:
            distinct_rows = gold_json_df.select(
                F.get_json_object(col("JSON_content"), path).alias("v")
            ).distinct().collect()
            offenders = sorted({(r.v if r.v is not None else "__NULL__") for r in distinct_rows
                                if r.v not in allowed})
            status = "PASS" if not offenders else "FAIL"
            msg = "" if not offenders else f"unexpected values: {offenders[:10]}"
            all_test_results.append(TestResult(test_name="test_when_map_value_ranges",
                                               test_field=path, test_from_state=state_under_test,
                                               status=status, message=msg))
        except Exception as e:
            all_test_results.append(TestResult(test_name="test_when_map_value_ranges",
                                               test_field=path, test_from_state=state_under_test,
                                               status="ERROR", message=f"{type(e).__name__}: {e}"))

_run_timed("test_when_map_value_ranges", test_when_map_value_ranges, gold_json)

## Test 7 — Re-do the code→label translation ourselves and check

For sampled cases, the cell below does two things:
- **WHEN-map round-trip:** reads the raw bronze code (Interpreter, HumanRights, AppealReceivedBy, etc.), applies our own Python mapping (defined right in this cell — *the test IS the spec*), and checks the result matches the JSON value. If the dev silently changes a mapping, the test breaks immediately.
- **Concat round-trip from silver:** rebuilds `AppellantFullName` (from Name/Forenames/Title) and `fileLocation` (from Centre/Department/Note) and checks the result matches the JSON.

If bronze isn't loadable in this env, the WHEN-map round-trip reports NO_DATA for those fields — the silver concat check still runs.

In [0]:
# Our reverse-engineered WHEN maps. Source: silver_appealcase_detail logic.
INTERPRETER_MAP       = {1: "YES", 2: "NO"}
ADDITIONALGROUNDS_MAP = {0: "", 1: "YES", 2: "NO"}
APPEALCATEGORIES_MAP  = {1: "YES", 2: "NO"}
HUMANRIGHTS_MAP       = {0: "", 1: "YES", 2: "NO"}
APPEALRECEIVEDBY_MAP  = {1: "Fax", 2: "Fax and Post", 3: "Hand", 4: "Post", 5: "On-Line", 0: ""}
CRRESPONDENT_MAP      = {1: "respondent", 2: "embassy", 3: "POU"}
LSCCOMMISSION_MAP     = {0: "", 1: "England & Wales", 2: "Northern Ireland", 3: "Scotland"}
VISITVISATYPE_MAP     = {0: "", 1: "On Papers", 2: "Oral Hearing"}

WHEN_MAP_FIELDS = [
    ("Interpreter",      INTERPRETER_MAP),
    ("VisitVisaType",    VISITVISATYPE_MAP),
    ("AdditionalGrounds",ADDITIONALGROUNDS_MAP),
    ("AppealCategories", APPEALCATEGORIES_MAP),
    ("HumanRights",      HUMANRIGHTS_MAP),
    ("AppealReceivedBy", APPEALRECEIVEDBY_MAP),
    ("CRRespondent",     CRRESPONDENT_MAP),
    ("LSCCommission",    LSCCOMMISSION_MAP),
]

def test_when_map_roundtrip():
    if bronze_appealcase is None:
        for field, _ in WHEN_MAP_FIELDS:
            all_test_results.append(TestResult(test_name="test_when_map_roundtrip",
                                               test_field=field, test_from_state=state_under_test,
                                               status="NO_DATA", message="bronze_appealcase not loadable"))
        return
    if not SAMPLE_CASES:
        return
    try:
        raw_cols = ["CaseNo"] + [f for f, _ in WHEN_MAP_FIELDS if f in bronze_appealcase.columns]
        raw_rows = bronze_appealcase.filter(col("CaseNo").isin(*SAMPLE_CASES)).select(*raw_cols).collect()
        raw_by_case = {r.CaseNo: r.asDict() for r in raw_rows}
    except Exception as e:
        for field, _ in WHEN_MAP_FIELDS:
            all_test_results.append(TestResult(test_name="test_when_map_roundtrip",
                                               test_field=field, test_from_state=state_under_test,
                                               status="ERROR", message=f"raw read failed: {e}"))
        return

    for field, mapping in WHEN_MAP_FIELDS:
        fails, errors, passes = 0, 0, 0
        examples = []
        if field not in bronze_appealcase.columns:
            all_test_results.append(TestResult(test_name="test_when_map_roundtrip",
                                               test_field=field, test_from_state=state_under_test,
                                               status="NO_DATA", message="raw column not present in bronze"))
            continue
        for case_no in SAMPLE_CASES:
            obj = SAMPLE_JSON.get(case_no)
            raw = raw_by_case.get(case_no)
            if obj is None or raw is None:
                continue
            try:
                raw_v = raw.get(field)
                expected = mapping.get(raw_v, "") if raw_v is not None else ""
                actual = obj.get(field)
                if (expected or "") == (actual or ""):
                    passes += 1
                else:
                    fails += 1
                    if len(examples) < 3:
                        examples.append(f"{case_no}: raw={raw_v} expected={expected!r} actual={actual!r}")
            except Exception as e:
                errors += 1
        status = "PASS" if fails == 0 and errors == 0 and passes > 0 else ("ERROR" if errors and not fails else ("FAIL" if fails else "NO_DATA"))
        msg = f"pass={passes} fail={fails} err={errors}" + ("; " + "; ".join(examples) if examples else "")
        all_test_results.append(TestResult(test_name="test_when_map_roundtrip",
                                           test_field=field, test_from_state=state_under_test,
                                           status=status, message=msg))


def _expected_appellant_full_name(name, forenames, title):
    parts = []
    if name:      parts.append(str(name))
    if forenames: parts.append(str(forenames))
    if title:     parts.append(f"({title})")
    return " ".join(parts)

def _expected_file_location(centre, department, note):
    return ", ".join([x for x in (centre, department, note) if x])

def test_concat_field_roundtrip():
    if not SAMPLE_CASES:
        return

    # Pull silver components in one shot
    try:
        app_rows = silver_applicant.filter(col("CaseNo").isin(*SAMPLE_CASES))             .select("CaseNo", "AppellantName", "AppellantForenames", "AppellantTitle").collect()
        app_by_case = {r.CaseNo: r for r in app_rows}
    except Exception as e:
        all_test_results.append(TestResult(test_name="test_concat_field_roundtrip",
                                           test_field="AppellantFullName", test_from_state=state_under_test,
                                           status="ERROR", message=f"silver_applicant read failed: {e}"))
        app_by_case = {}

    try:
        appcase_rows = silver_appealcase.filter(col("CaseNo").isin(*SAMPLE_CASES))             .select("CaseNo", "FileLocationCentre", "FileLocationDepartment", "FileLocationNote").collect()
        appcase_by_case = {r.CaseNo: r for r in appcase_rows}
    except Exception as e:
        all_test_results.append(TestResult(test_name="test_concat_field_roundtrip",
                                           test_field="fileLocation", test_from_state=state_under_test,
                                           status="ERROR", message=f"silver_appealcase read failed: {e}"))
        appcase_by_case = {}

    # AppellantFullName
    fails, passes, examples = 0, 0, []
    for case_no in SAMPLE_CASES:
        obj = SAMPLE_JSON.get(case_no)
        a = app_by_case.get(case_no)
        if obj is None or a is None:
            continue
        expected = _expected_appellant_full_name(a.AppellantName, a.AppellantForenames, a.AppellantTitle)
        actual = obj.get("AppellantFullName")
        if (expected or "") == (actual or ""):
            passes += 1
        else:
            fails += 1
            if len(examples) < 3:
                examples.append(f"{case_no}: expected={expected!r} actual={actual!r}")
    status = "PASS" if fails == 0 and passes > 0 else ("FAIL" if fails else "NO_DATA")
    msg = f"pass={passes} fail={fails}" + ("; " + "; ".join(examples) if examples else "")
    all_test_results.append(TestResult(test_name="test_concat_field_roundtrip",
                                       test_field="AppellantFullName", test_from_state=state_under_test,
                                       status=status, message=msg))

    # fileLocation
    fails, passes, examples = 0, 0, []
    for case_no in SAMPLE_CASES:
        obj = SAMPLE_JSON.get(case_no)
        a = appcase_by_case.get(case_no)
        if obj is None or a is None:
            continue
        expected = _expected_file_location(a.FileLocationCentre, a.FileLocationDepartment, a.FileLocationNote)
        actual = obj.get("fileLocation")
        if (expected or "") == (actual or ""):
            passes += 1
        else:
            fails += 1
            if len(examples) < 3:
                examples.append(f"{case_no}: expected={expected!r} actual={actual!r}")
    status = "PASS" if fails == 0 and passes > 0 else ("FAIL" if fails else "NO_DATA")
    msg = f"pass={passes} fail={fails}" + ("; " + "; ".join(examples) if examples else "")
    all_test_results.append(TestResult(test_name="test_concat_field_roundtrip",
                                       test_field="fileLocation", test_from_state=state_under_test,
                                       status=status, message=msg))

_run_timed("test_when_map_roundtrip", test_when_map_roundtrip)
_run_timed("test_concat_field_roundtrip", test_concat_field_roundtrip)

## Test 8 — Do pass-through fields come straight through from bronze?

Some fields in the JSON aren't transformed by silver — they pass through bronze→silver→gold unchanged (CaseNo, CasePrefix, dates, IDs, names, addresses, port, nationality, etc.). The cell below compares the bronze value directly to the JSON value for sampled cases. Catches silent corruption between bronze and gold that other tests would miss.

Dates are compared on the `YYYY-MM-DD` prefix (timezone-tolerant). Numerics use a float-equal fallback so `2015` vs `'2015'` match.

In [0]:
# Source of truth: silver_appealcase_detail copies these from bronze with no transformation.
BRONZE_APPEALCASE_PASSTHROUGH = [
    "CaseNo", "CasePrefix", "CaseYear", "CaseType",
    "AppealTypeId", "PortId", "HORef", "NationalityId", "Nationality",
    "CountryId", "CountryOfTravelOrigin", "ThirdCountryId", "ThirdCountry",
    "HOInterpreter", "AppealCaseNote", "UserId", "CaseOutcomeId",
    "CaseFeeSummaryId", "RepresentativeRef", "Language",
]
BRONZE_APPEALCASE_PASSTHROUGH_DATES = [
    "DateLodged", "DateServed", "DateAppealReceived", "DateOfIssue",
    "DateApplicationLodged", "DateOfApplicationDecision", "FileLocationTransferDate",
    "ProvisionalDestructionDate",
]
BRONZE_APPLICANT_PASSTHROUGH = [
    "AppellantId", "AppellantName", "AppellantForenames", "AppellantTitle",
    "AppellantAddress1", "AppellantAddress2", "AppellantAddress3",
    "AppellantAddress4", "AppellantPostcode", "PortReference",
]
BRONZE_APPLICANT_PASSTHROUGH_DATES = ["AppellantBirthDate"]

def _values_match(a, b, is_date=False):
    if a is None and b is None: return True
    if a is None or b is None:
        if a == "" or b == "": return True   # treat null vs empty-string as equal
        return False
    sa, sb = str(a).strip(), str(b).strip()
    if sa == sb: return True
    if is_date and len(sa) >= 10 and len(sb) >= 10:
        return sa[:10] == sb[:10]
    try:
        if float(sa) == float(sb): return True
    except (ValueError, TypeError):
        pass
    return False

def _check_passthrough(bronze_df, fields_scalar, fields_date, extra_filter=None, label=""):
    if bronze_df is None or not SAMPLE_CASES:
        for f in (fields_scalar + fields_date):
            all_test_results.append(TestResult(test_name="test_bronze_passthrough_scalars",
                                               test_field=f, test_from_state=state_under_test,
                                               status="NO_DATA", message=f"{label} not loaded"))
        return
    available_scalar = [f for f in fields_scalar if f in bronze_df.columns]
    available_dates  = [f for f in fields_date   if f in bronze_df.columns]
    all_fields = available_scalar + available_dates
    if not all_fields:
        return
    df = bronze_df.filter(col("CaseNo").isin(*SAMPLE_CASES))
    if extra_filter is not None:
        df = df.filter(extra_filter)
    try:
        rows = df.select("CaseNo", *all_fields).collect()
    except Exception as e:
        for f in all_fields:
            all_test_results.append(TestResult(test_name="test_bronze_passthrough_scalars",
                                               test_field=f, test_from_state=state_under_test,
                                               status="ERROR", message=f"bronze read failed: {e}"))
        return
    by_case = {r.CaseNo: r.asDict() for r in rows}

    for field in all_fields:
        is_date = field in fields_date
        passes, fails, examples = 0, 0, []
        for case_no in SAMPLE_CASES:
            obj = SAMPLE_JSON.get(case_no)
            row = by_case.get(case_no)
            if obj is None or row is None:
                continue
            bv = row.get(field)
            jv = obj.get(field)
            if _values_match(bv, jv, is_date=is_date):
                passes += 1
            else:
                fails += 1
                if len(examples) < 3:
                    examples.append(f"{case_no}: bronze={bv!r} json={jv!r}")
        if fails == 0 and passes == 0:
            status, msg = "NO_DATA", "no rows compared"
        elif fails == 0:
            status, msg = "PASS", f"pass={passes}"
        else:
            status, msg = "FAIL", f"pass={passes} fail={fails}; " + "; ".join(examples)
        all_test_results.append(TestResult(test_name="test_bronze_passthrough_scalars",
                                           test_field=field, test_from_state=state_under_test,
                                           status=status, message=msg))

def test_bronze_passthrough_scalars():
    _check_passthrough(bronze_appealcase,
                       BRONZE_APPEALCASE_PASSTHROUGH,
                       BRONZE_APPEALCASE_PASSTHROUGH_DATES,
                       extra_filter=None,
                       label="bronze_appealcase")
    # Primary appellant only (silver_applicant_detail filters CaseAppellantRelationship IS NULL)
    extra = col("CaseAppellantRelationship").isNull() if (bronze_applicant is not None and "CaseAppellantRelationship" in bronze_applicant.columns) else None
    _check_passthrough(bronze_applicant,
                       BRONZE_APPLICANT_PASSTHROUGH,
                       BRONZE_APPLICANT_PASSTHROUGH_DATES,
                       extra_filter=extra,
                       label="bronze_applicant")

_run_timed("test_bronze_passthrough_scalars", test_bronze_passthrough_scalars)

## Test 9 — Do JSON arrays match bronze counts?

Same idea as Test 4 (silver count vs JSON length) but using bronze. Catches data loss between bronze and silver — a bug Test 4 would miss because if silver lost a row both its count AND the JSON's length would shrink in lockstep, and Test 4 would still pass.

Note: some bronze tables are pre-joined chunks (e.g. ListDetails carries one row per case×adjudicator×court combination), so a single ListId can legitimately appear multiple times in bronze AND in the JSON's ListDetails array — both counts should match as-is.

In [0]:
BRONZE_ARRAY_COUNTS = [
    # (json_key, bronze_var_name)
    ("HistoryDetails",     "bronze_history"),
    ("StatusDetails",      "bronze_status"),
    ("ListDetails",        "bronze_list"),
    ("TransactionDetails", "bronze_transaction"),
    ("DocumentDetails",    "bronze_documents"),
]

def test_bronze_array_row_counts():
    if not SAMPLE_CASES:
        return
    bronze_counts = {}
    for json_key, bronze_var in BRONZE_ARRAY_COUNTS:
        df = BRONZE_BY_NAME.get(bronze_var)
        if df is None:
            bronze_counts[bronze_var] = None
            continue
        try:
            cnt = df.filter(col("CaseNo").isin(*SAMPLE_CASES)).groupBy("CaseNo").count().collect()
            bronze_counts[bronze_var] = {r.CaseNo: r["count"] for r in cnt}
        except Exception as e:
            bronze_counts[bronze_var] = None
            print(f"bronze count for {bronze_var} failed: {e}")

    for case_no in SAMPLE_CASES:
        obj = SAMPLE_JSON.get(case_no)
        if obj is None:
            continue
        for json_key, bronze_var in BRONZE_ARRAY_COUNTS:
            try:
                bc = bronze_counts.get(bronze_var)
                if bc is None:
                    all_test_results.append(TestResult(test_name="test_bronze_array_row_counts",
                                                       test_field=f"{case_no}:{json_key}",
                                                       test_from_state=state_under_test,
                                                       status="NO_DATA", message=f"{bronze_var} not loaded"))
                    continue
                arr = obj.get(json_key) or []
                json_n = len(arr) if isinstance(arr, list) else -1
                bronze_n = bc.get(case_no, 0)
                ok = json_n == bronze_n
                all_test_results.append(TestResult(test_name="test_bronze_array_row_counts",
                                                   test_field=f"{case_no}:{json_key}",
                                                   test_from_state=state_under_test,
                                                   status=("PASS" if ok else "FAIL"),
                                                   message=("" if ok else f"json={json_n} bronze={bronze_n}")))
            except Exception as e:
                all_test_results.append(TestResult(test_name="test_bronze_array_row_counts",
                                                   test_field=f"{case_no}:{json_key}",
                                                   test_from_state=state_under_test,
                                                   status="ERROR", message=f"{type(e).__name__}: {e}"))

_run_timed("test_bronze_array_row_counts", test_bronze_array_row_counts)

## Test 10 — Re-do the concat fields from bronze and check

Same as Test 7's concat round-trip, but the cell below pulls name and location components from bronze instead of silver. Useful diagnostic: if Test 7 (silver) passes but this test (bronze) fails, the bronze→silver step is the breaker; if Test 7 also fails, the bug is shared.

In [0]:
def test_bronze_concat_roundtrip():
    if not SAMPLE_CASES:
        return

    # AppellantFullName — from bronze_applicant filtered to primary appellant only
    if bronze_applicant is None:
        all_test_results.append(TestResult(test_name="test_bronze_concat_roundtrip",
                                           test_field="AppellantFullName", test_from_state=state_under_test,
                                           status="NO_DATA", message="bronze_applicant not loaded"))
    else:
        try:
            cols_needed = [c for c in ("AppellantName", "AppellantForenames", "AppellantTitle") if c in bronze_applicant.columns]
            df = bronze_applicant.filter(col("CaseNo").isin(*SAMPLE_CASES))
            if "CaseAppellantRelationship" in bronze_applicant.columns:
                df = df.filter(col("CaseAppellantRelationship").isNull())
            rows = df.select("CaseNo", *cols_needed).collect()
            apl_by_case = {r.CaseNo: r for r in rows}
            passes, fails, examples = 0, 0, []
            for case_no in SAMPLE_CASES:
                obj = SAMPLE_JSON.get(case_no)
                a = apl_by_case.get(case_no)
                if obj is None or a is None: continue
                expected = _expected_appellant_full_name(
                    getattr(a, "AppellantName", None),
                    getattr(a, "AppellantForenames", None),
                    getattr(a, "AppellantTitle", None),
                )
                actual = obj.get("AppellantFullName")
                if (expected or "") == (actual or ""):
                    passes += 1
                else:
                    fails += 1
                    if len(examples) < 3:
                        examples.append(f"{case_no}: expected={expected!r} actual={actual!r}")
            status = "PASS" if fails == 0 and passes > 0 else ("FAIL" if fails else "NO_DATA")
            msg = f"pass={passes} fail={fails}" + ("; " + "; ".join(examples) if examples else "")
            all_test_results.append(TestResult(test_name="test_bronze_concat_roundtrip",
                                               test_field="AppellantFullName", test_from_state=state_under_test,
                                               status=status, message=msg))
        except Exception as e:
            all_test_results.append(TestResult(test_name="test_bronze_concat_roundtrip",
                                               test_field="AppellantFullName", test_from_state=state_under_test,
                                               status="ERROR", message=f"{type(e).__name__}: {e}"))

    # fileLocation — from bronze_appealcase
    if bronze_appealcase is None:
        all_test_results.append(TestResult(test_name="test_bronze_concat_roundtrip",
                                           test_field="fileLocation", test_from_state=state_under_test,
                                           status="NO_DATA", message="bronze_appealcase not loaded"))
        return
    try:
        cols_needed = [c for c in ("FileLocationCentre", "FileLocationDepartment", "FileLocationNote") if c in bronze_appealcase.columns]
        rows = bronze_appealcase.filter(col("CaseNo").isin(*SAMPLE_CASES)).select("CaseNo", *cols_needed).collect()
        apc_by_case = {r.CaseNo: r for r in rows}
        passes, fails, examples = 0, 0, []
        for case_no in SAMPLE_CASES:
            obj = SAMPLE_JSON.get(case_no)
            a = apc_by_case.get(case_no)
            if obj is None or a is None: continue
            expected = _expected_file_location(
                getattr(a, "FileLocationCentre", None),
                getattr(a, "FileLocationDepartment", None),
                getattr(a, "FileLocationNote", None),
            )
            actual = obj.get("fileLocation")
            if (expected or "") == (actual or ""):
                passes += 1
            else:
                fails += 1
                if len(examples) < 3:
                    examples.append(f"{case_no}: expected={expected!r} actual={actual!r}")
        status = "PASS" if fails == 0 and passes > 0 else ("FAIL" if fails else "NO_DATA")
        msg = f"pass={passes} fail={fails}" + ("; " + "; ".join(examples) if examples else "")
        all_test_results.append(TestResult(test_name="test_bronze_concat_roundtrip",
                                           test_field="fileLocation", test_from_state=state_under_test,
                                           status=status, message=msg))
    except Exception as e:
        all_test_results.append(TestResult(test_name="test_bronze_concat_roundtrip",
                                           test_field="fileLocation", test_from_state=state_under_test,
                                           status="ERROR", message=f"{type(e).__name__}: {e}"))

_run_timed("test_bronze_concat_roundtrip", test_bronze_concat_roundtrip)

## Test 11 — Deep set-equality on key fields per case

The cheapest checks count rows; the most thorough check compares actual primary-key values. For 5 deeply-sampled cases, the cell below pulls the SET of key tuples (HistoryId, StatusId, ListId×StatusId, TransactionId, ReceivedDocumentId) from silver and the same set from the JSON array, then compares. Catches missing entries, extra entries, AND any key-field corruption.

In [0]:
DEEP_KEY_BY_ARRAY = [
    # (json_key, silver_var_name, key_columns_tuple)
    ("HistoryDetails",     "silver_history",     ("HistoryId",)),
    ("StatusDetails",      "silver_status",      ("StatusId",)),
    ("ListDetails",        "silver_list",        ("ListId", "StatusId")),
    ("TransactionDetails", "silver_transaction", ("TransactionId",)),
    ("DocumentDetails",    "silver_documents",   ("ReceivedDocumentId",)),
]

def test_sampled_deep_field_equality(n=5):
    deep_sample = SAMPLE_CASES[:n]
    if not deep_sample:
        return
    for case_no in deep_sample:
        obj = SAMPLE_JSON.get(case_no)
        if obj is None:
            continue
        for json_key, silver_var, key_cols in DEEP_KEY_BY_ARRAY:
            try:
                arr = obj.get(json_key) or []
                json_keys = set(tuple(d.get(k) for k in key_cols) for d in arr if isinstance(d, dict))

                df = SILVER_BY_NAME[silver_var]
                agg_col = SILVER_AGGREGATED_ARRAY_COL.get(silver_var)
                if agg_col and agg_col in df.columns:
                    # silver is pre-aggregated: explode the nested array, then read key fields
                    silver_rows = df.filter(col("CaseNo") == case_no)                                     .select(F.explode(col(agg_col)).alias("e"))                                     .select(*[col(f"e.{k}").alias(k) for k in key_cols]).collect()
                else:
                    silver_rows = df.filter(col("CaseNo") == case_no).select(*key_cols).collect()
                silver_keys = set(tuple(getattr(r, k, None) for k in key_cols) for r in silver_rows)

                missing = sorted(str(k) for k in (silver_keys - json_keys))
                extra   = sorted(str(k) for k in (json_keys - silver_keys))
                ok = not missing and not extra
                msg = "" if ok else f"missing={missing[:5]} extra={extra[:5]}"
                all_test_results.append(TestResult(test_name="test_sampled_deep_field_equality",
                                                   test_field=f"{case_no}:{json_key}",
                                                   test_from_state=state_under_test,
                                                   status=("PASS" if ok else "FAIL"), message=msg))
            except Exception as e:
                all_test_results.append(TestResult(test_name="test_sampled_deep_field_equality",
                                                   test_field=f"{case_no}:{json_key}",
                                                   test_from_state=state_under_test,
                                                   status="ERROR", message=f"{type(e).__name__}: {e}"))

_run_timed("test_sampled_deep_field_equality", test_sampled_deep_field_equality, 5)

In [0]:
elapsed = perf_time.time() - t0
counts = count_by_status(all_test_results)

print(f"Tests complete in {elapsed:.1f}s")
print(f"  Total : {counts['TOTAL']}")
print(f"  Pass  : {counts['PASS']}")
print(f"  Fail  : {counts['FAIL']}")
print(f"  NoData: {counts['NO_DATA']}")
print(f"  Error : {counts['ERROR']}")

# Plain-English description per test_name. Goes into the `description` column of
# the CSV and test_automation_results2 so the results are self-explanatory.
DESCRIPTIONS = {'test_json_parseable_and_caseno_matches': "Test 1: Every gold row's JSON_content is valid JSON, and its top-level CaseNo matches the row's CaseNo column.", 'test_no_failure_json_content': "Test 1: No gold row has JSON_content starting with 'failure' (pipeline error markers that should have been dropped before gold).", 'test_required_scalar_keys': 'Test 2: This required top-level scalar key is present (non-null) on every gold row.', 'test_informational_scalar_keys': 'Test 2: This scalar key is usually populated but legitimately NULL on some legacy data (e.g. pre-2000 cases). Reports a count, never FAIL.', 'test_expected_array_keys': 'Test 2: Reports how many rows have this array key present. Absence is normal when the case has zero source rows for the array.', 'test_array_absences_match_silver': 'Test 3: When the JSON omits this array key, silver has zero source rows for that case. A real bug is silver having rows that the JSON dropped.', 'test_array_cardinality_vs_silver': "Test 4: For this case, the JSON array length equals the count of silver rows for the array's source table.", 'test_primary_appellant_partition': "Test 5: The JSON's AppellantId is in silver_applicant_detail (primary appellants only) and NOT in silver_dependent_detail.", 'test_dependent_partition': "Test 5: Every entry in the JSON's DependentDetails is in silver_dependent_detail and NOT in silver_applicant_detail.", 'test_when_map_value_ranges': 'Test 6: Every distinct JSON value at this path is in the documented set of mapped labels (e.g. YES/NO, checked/disabled).', 'test_when_map_roundtrip': 'Test 7: For sampled cases — raw bronze code, run through our own Python mapping, equals the JSON value.', 'test_concat_field_roundtrip': 'Test 7: Field rebuilt from silver name/location components matches the JSON value (sampled cases).', 'test_bronze_passthrough_scalars': "Test 8: Bronze value for this pass-through field equals the JSON value (sampled cases). Catches corruption between bronze and gold for fields silver doesn't transform.", 'test_bronze_array_row_counts': "Test 9: For this case, the JSON array length equals the count of bronze rows for the array's source table. Catches data loss between bronze and silver.", 'test_bronze_concat_roundtrip': 'Test 10: Field rebuilt from bronze name/location components matches the JSON value (sampled cases).', 'test_sampled_deep_field_equality': 'Test 11: For this deeply-sampled case, the SET of key tuples (e.g. HistoryId) matches between silver and JSON. Catches missing/extra entries and key corruption.', '00_OVERALL_SUMMARY': 'Overall verdict for this run: total tests plus pass/fail/error/no-data counts.'}

# Enrich every result row with its description (unless one was set explicitly).
for r in all_test_results:
    if not getattr(r, "description", ""):
        r.description = DESCRIPTIONS.get(r.test_name, "")

# Headline summary row — sorted to the top of the CSV / audit table so the overall
# verdict is the first thing you see when opening the results.
overall_status = "FAIL" if (counts["FAIL"] + counts["ERROR"]) > 0 else "PASS"
all_test_results.append(TestResult(
    test_name="00_OVERALL_SUMMARY",
    test_field="00_OVERALL",
    test_from_state=state_under_test,
    status=overall_status,
    message=(f"Total={counts['TOTAL']} Pass={counts['PASS']} "
             f"Fail={counts['FAIL']} Error={counts['ERROR']} "
             f"NoData={counts['NO_DATA']} Elapsed={elapsed:.1f}s"),
    description=DESCRIPTIONS.get("00_OVERALL_SUMMARY", ""),
))

## Save results

The cell below sorts the result rows (overall headline first, then failures, then everything else alphabetically by test_field), writes a timestamped CSV under your `/Workspace/Users/<you>/Results/Archive_FPA_Tests/` folder, and appends rows to the three central audit tables:
- `test_automation_runs2` — one row per run with overall verdict.
- `test_automation_results2` — one row per individual test result, with the new `description` column.
- `test_automation_perfresults2` — one row per test function with elapsed time.

In [0]:
# Headline row always first; then fails to the top, then alphabetical by test_field.
status_order = {"FAIL": 0, "ERROR": 1, "NO_DATA": 2, "PASS": 3}
def _sort_key(r):
    if str(getattr(r, "test_name", "")) == "00_OVERALL_SUMMARY":
        return (-1, "")
    return (status_order.get(str(r.status).upper(), 4), str(r.test_field))
all_test_results.sort(key=_sort_key)

folder, timestamp, _ = build_report_folder(test_results_path, f"{state_under_test}_{run_tag}")
print(f"Reports folder: {folder}")

try:
    csv_path  = write_csv(all_test_results, state_under_test, folder, timestamp)
    print(f"CSV  : {csv_path}")
except Exception as e:
    print(f"write_csv FAILED: {e}")

try:
    run_id = create_run(
        spark,
        run_user=run_user,
        run_start_datetime=run_start_datetime,
        run_by_automation_name="Archive_FPA_JSON_Tests",
        run_tag=run_tag,
        run_status=("FAILED" if any(str(getattr(r, "status", "")).upper().strip() in ("FAIL", "ERROR") for r in all_test_results) else "PASSED"),
        state_under_test=state_under_test,
    )
    print(f"Created run_id={run_id}")
    n = create_results(spark, run_id, all_test_results)
    print(f"Wrote {n} result rows")
    p = create_perf_results(spark, run_id, perf_timings, state_under_test)
    print(f"Wrote {p} perf rows to test_automation_perfresults2")
except Exception as e:
    print(f"audit write FAILED: {e}")

## Final summary

The cell below prints a clear banner with the overall PASS/FAIL verdict and the totals. If anything failed, the first 10 failing rows are listed inline so you don't have to open the CSV to triage. The slowest test functions are also listed so you can see where time was spent.

In [0]:
failures = counts["FAIL"] + counts["ERROR"]
verdict = "PASS" if failures == 0 else "FAIL"
banner_line = "=" * 70
print(banner_line)
print(f"  OVERALL RESULT: {verdict}    ({state_under_test})")
print(banner_line)
print(f"  Total tests : {counts['TOTAL']}")
print(f"  Passed      : {counts['PASS']}")
print(f"  Failed      : {counts['FAIL']}")
print(f"  Errored     : {counts['ERROR']}")
print(f"  No data     : {counts['NO_DATA']}")
print(f"  Elapsed     : {elapsed:.1f}s")
print(banner_line)
if verdict == "FAIL":
    print(f"  >>> {failures} failure(s) — see CSV / test_automation_results2 for detail.")
    fail_examples = [(r.test_name, r.test_field, r.message) for r in all_test_results
                     if str(getattr(r, "status", "")).upper().strip() in ("FAIL", "ERROR")][:10]
    for tn, tf, msg in fail_examples:
        print(f"    [{tn}] {tf} -- {msg[:120]}")
else:
    print(f"  >>> All {counts['TOTAL']} tests passed.")

# Performance summary — top 5 slowest tests
print()
print("PERF: slowest tests")
print("-" * 70)
slowest = sorted(perf_timings, key=lambda p: p["elapsed_seconds"], reverse=True)[:5]
for p in slowest:
    print(f"  {p['elapsed_seconds']:7.2f}s  +{p['result_count']:4d} results  {p['test_name']}")
total_test_time = sum(p["elapsed_seconds"] for p in perf_timings)
print(f"  ------- ")
print(f"  {total_test_time:7.2f}s  total time across {len(perf_timings)} test functions")
print(f"  Full perf timings: test_automation_perfresults2 (run_id={run_id if 'run_id' in dir() else '?'})")